In [1]:
import pandas as pd
import talib.abstract as ta
import numpy as np

In [8]:
df = pd.read_csv('bitcoin.csv')
tdf = df.tail(1000)
tdf.tail(100)

,Date,Open,High,Low,Close,Volume
36068,2021-10-03 06:00:00,47990.13,48057.00,47896.93,47972.99,617.25174
36069,2021-10-03 07:00:00,47970.15,47970.16,47814.00,47895.99,569.31948
36070,2021-10-03 08:00:00,47896.00,47983.38,47798.50,47823.99,838.17079
36071,2021-10-03 09:00:00,47823.98,47869.00,47600.00,47830.09,1489.72226
36072,2021-10-03 10:00:00,47830.09,47953.20,47730.00,47812.32,625.24084
...,...,...,...,...,...,...
36163,2021-10-07 05:00:00,55073.20,55073.21,54545.07,54735.76,2251.12202
36164,2021-10-07 06:00:00,54735.77,54968.06,54375.83,54534.16,1783.00426
36165,2021-10-07 07:00:00,54534.16,54793.26,54235.33,54755.92,4163.43136
36166,2021-10-07 08:00:00,54755.91,54778.91,54400.00,54538.30,2049.38218


In [9]:
tdf.columns=[col.lower() for col in tdf.columns]
tdf

,date,open,high,low,close,volume
35168,2021-08-26 16:00:00,46800.00,47056.86,46485.16,46897.72,2263.61921
35169,2021-08-26 17:00:00,46897.72,47123.65,46886.01,46950.26,1276.36545
35170,2021-08-26 18:00:00,46950.26,47092.63,46731.00,47001.46,1128.54621
35171,2021-08-26 19:00:00,47001.47,47043.88,46755.27,46939.75,1212.66545
35172,2021-08-26 20:00:00,46934.22,47058.23,46845.00,47017.57,949.73043
...,...,...,...,...,...,...
36163,2021-10-07 05:00:00,55073.20,55073.21,54545.07,54735.76,2251.12202
36164,2021-10-07 06:00:00,54735.77,54968.06,54375.83,54534.16,1783.00426
36165,2021-10-07 07:00:00,54534.16,54793.26,54235.33,54755.92,4163.43136
36166,2021-10-07 08:00:00,54755.91,54778.91,54400.00,54538.30,2049.38218


In [17]:
def regression_ema(dataframe:pd.DataFrame, timeperiod=144) -> pd.Series:
    df = dataframe.copy()
    # get deviation
    df['cline'] = ta.MA(dataframe, 50)
    df['deviation'] = (df['close']-df['cline']) 
    # get percentile
    df['bull_quantile'] = df['deviation'].mask(df['deviation'] < 0).rolling(window=timeperiod).quantile(0.6)
    df['bear_quantile'] = df['deviation'].mask(df['deviation'] > 0).rolling(window=timeperiod).quantile(0.4)
    # df['pass_reg_filter'] = np.where(
    #     df['deviation'].between(df['bull_quantile'], df['bear_quantile']), 1, 0)
    df['pass_reg_filter'] = np.where(
        ~pd.isna(df['deviation']),
        np.where(
        df['deviation'] > df['bull_quantile'], -1, 
        np.where((df['deviation'] < df['bear_quantile']), 1, 0)),
        np.nan
        )
    return df[['deviation','bull_quantile', 'bear_quantile', 'pass_reg_filter']]

In [19]:
tdf[['deviation', 'bull_quantile', 'bear_quantile', 'pass_reg_filter']] = regression_ema(tdf, 21)

C:\Users\xlfin\AppData\Local\Temp\ipykernel_12964\3930854815.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tdf[['deviation', 'bull_quantile', 'bear_quantile', 'pass_reg_filter']] = regression_ema(tdf, 21)
C:\Users\xlfin\AppData\Local\Temp\ipykernel_12964\3930854815.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tdf[['deviation', 'bull_quantile', 'bear_quantile', 'pass_reg_filter']] = regression_ema(tdf, 21)
C:\Users\xlfin\AppData\Local\Temp\ipykernel_12964\3930854815.py:1: SettingWithCopyWarning:

In [21]:
tdf.tail(50)

,date,open,high,low,close,volume,deviation,bull_quantile,bear_quantile,pass_reg_filter
36118,2021-10-05 08:00:00,49476.52,50000.00,49376.06,49919.99,2544.685460,-80.01,NaN,-831.69,0.0
36119,2021-10-05 09:00:00,49920.00,50328.00,49584.40,50269.35,3724.600840,269.35,NaN,NaN,0.0
36120,2021-10-05 10:00:00,50269.35,50350.00,49839.26,49999.99,2413.651530,-0.01,NaN,NaN,0.0
36121,2021-10-05 11:00:00,49999.99,50145.88,49801.74,49877.06,1611.669930,-122.94,NaN,NaN,0.0
36122,2021-10-05 12:00:00,49877.06,50110.00,49705.00,50000.00,2161.324870,0.00,NaN,NaN,0.0
36123,2021-10-05 13:00:00,50000.00,50320.00,49772.63,50207.96,2108.424410,207.96,NaN,NaN,0.0
36124,2021-10-05 14:00:00,50207.96,50388.00,49575.94,49841.79,3331.248360,-158.21,NaN,NaN,0.0
36125,2021-10-05 15:00:00,49841.78,50205.21,49700.00,49793.57,2394.236910,-206.43,NaN,NaN,0.0
36126,2021-10-05 16:00:00,49793.57,50115.24,49640.36,50103.85,1616.066060,103.85,NaN,NaN,0.0
36127,2021-10-05 17:00:00,50103.86,50200.00,49950.00,50146.52,1496.736340,146.52,NaN,NaN,0.0
